# Module 3 — Lab D: Model Training with MLflow Experiment Tracking
## Train Classification Models, Compare Metrics & Register the Best Model

---

### Lab Overview

With the feature matrix produced in **Lab C** (`final_features.csv`, ~12,308 rows x 37 columns), you will now **train three classification models**, evaluate them on multiple metrics, and **track every experiment in MLflow** running on your EC2 instance.

FreshBasket's logistics team (Priya is the ML lead, Arjun handles DevOps) needs a model that predicts whether a truck delivery will be **delayed** (target = 1) or **on-time** (target = 0). A single accuracy number is not enough — the operations team cares about **recall** (catching delayed shipments) while finance cares about **precision** (avoiding false alarms that trigger costly rerouting at ₹8,000 per unnecessary intervention).

| Step | What We Do | Key Technology |
|------|-----------|----------------|
| 1 | Environment setup + MLflow connection | `mlflow`, `boto3` |
| 2 | Load `final_features.csv` from S3 or local | `pandas`, `boto3` |
| 3 | Train / Validation / Test split (70/15/15) | `train_test_split` (stratified) |
| 4 | Preprocess: OneHotEncode + StandardScale | `OneHotEncoder`, `StandardScaler` |
| 5 | Train Logistic Regression (baseline) | `LogisticRegression` + MLflow |
| 6 | Train Random Forest | `RandomForestClassifier` + MLflow |
| 7 | Train XGBoost | `XGBClassifier` + MLflow |
| 8 | Compare all models side-by-side | Comparison DataFrame + bar chart |
| 9 | Final evaluation on held-out test set | Best model on unseen data |
| 10 | Register best model in MLflow Model Registry | `mlflow.register_model()` |
| 11 | Save artifacts to S3 | `boto3` upload |

### Learning Objectives

By the end of this lab you will be able to:

1. Connect a Jupyter notebook to a **self-hosted MLflow Tracking Server** on EC2.
2. Train and evaluate **Logistic Regression**, **Random Forest**, and **XGBoost** classifiers.
3. Log **parameters, metrics, artifacts, and models** to MLflow for every experiment run.
4. Interpret classification metrics — **Accuracy, Precision, Recall, F1 Score, ROC-AUC** — and explain what each means for the business.
5. **Register** the best-performing model in the MLflow Model Registry and transition it to **Staging**.
6. Upload trained model artifacts (model, encoder, scaler) to **S3** for downstream use in Lab E.

### Artifact Chain

```
Lab C  --------------------------->  Lab D  --------------------------->  Lab E
saves: final_features.csv (S3)      saves: xgboost_model.pkl           loads: model + encoder + scaler
                                           encoder.pkl                  Streamlit dashboard
                                           scaler.pkl                   Batch scoring pipeline
                                           model_metadata.json
                                    logs:  3 MLflow experiment runs
```

---

## 1. Environment Setup

> **SageMaker Notebook:** Run the cell below as-is — most packages are pre-installed.
>
> **Local Jupyter:** Uncomment the `pip install` line and run once.
>
> **Google Colab:** Uncomment the `pip install` line and run once.

In [ ]:
# -- Uncomment the line below if running locally or on Colab --
# !pip install pandas numpy matplotlib seaborn scikit-learn xgboost mlflow boto3 joblib -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report, RocCurveDisplay)
import xgboost as xgb
import mlflow
import mlflow.sklearn
import mlflow.xgboost
import joblib
import boto3
import json
import os
import warnings
warnings.filterwarnings('ignore')

# -- Display settings --
pd.set_option('display.max_columns', 40)
plt.rcParams['figure.figsize'] = (10, 6)
sns.set_style('whitegrid')

print("All libraries imported successfully")

In [ ]:
# ================================================================
#  MLflow Configuration
# ================================================================
# Replace <EC2_PUBLIC_IP> with your EC2 instance's public IP address.
# Example: "http://54.210.85.123:5000"
# If running MLflow locally: "http://127.0.0.1:5000"

MLFLOW_TRACKING_URI = "http://<EC2_PUBLIC_IP>:5000"   # <-- EDIT THIS
EXPERIMENT_NAME     = "truck-delay-classification"

# -- S3 Configuration --
S3_BUCKET    = "freshbasket-mlops"         # <-- EDIT if different
S3_DATA_KEY  = "data/final_features.csv"
S3_MODEL_DIR = "models/truck-delay/"

# -- Try connecting to MLflow; fall back to local if unreachable --
try:
    mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
    mlflow.set_experiment(EXPERIMENT_NAME)
    # Quick connectivity test
    mlflow.search_experiments()
    print(f"Connected to MLflow server: {MLFLOW_TRACKING_URI}")
    print(f"Experiment: '{EXPERIMENT_NAME}'")
    MLFLOW_AVAILABLE = True
except Exception as e:
    print(f"Could not reach MLflow at {MLFLOW_TRACKING_URI}")
    print(f"Error: {e}")
    print("Falling back to local MLflow tracking (./mlruns)")
    mlflow.set_tracking_uri("mlruns")
    mlflow.set_experiment(EXPERIMENT_NAME)
    MLFLOW_AVAILABLE = False

print(f"\nMLflow tracking URI: {mlflow.get_tracking_uri()}")

## 2. Load Feature Matrix

The feature matrix was created in **Lab C** and saved to S3 as `final_features.csv`. It contains **~12,308 rows** (one per scheduled truck trip) and **37 columns** (36 features + 1 target called `delay`).

We try to load from S3 first. If that fails (e.g., you are offline or the bucket is not set up yet), we fall back to a local copy.

In [ ]:
# -- Load final_features.csv from S3 or local fallback --
LOCAL_PATH = "final_features.csv"

try:
    s3 = boto3.client('s3')
    s3.download_file(S3_BUCKET, S3_DATA_KEY, LOCAL_PATH)
    print(f"Downloaded from S3: s3://{S3_BUCKET}/{S3_DATA_KEY}")
except Exception as e:
    print(f"S3 download failed: {e}")
    print(f"Looking for local file: {LOCAL_PATH}")
    if not os.path.exists(LOCAL_PATH):
        raise FileNotFoundError(
            f"{LOCAL_PATH} not found locally either.\n"
            "Run Lab C first, or upload final_features.csv to this directory."
        )

df = pd.read_csv(LOCAL_PATH)
print(f"\nLoaded: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"\nColumn dtypes:\n{df.dtypes.value_counts()}")

In [ ]:
# -- Sanity checks --
print("=" * 55)
print("  DATA SANITY CHECKS")
print("=" * 55)

# Target distribution
print(f"\n1. Target variable -- 'delay':")
target_counts = df['delay'].value_counts()
print(f"   On-Time (0): {target_counts.get(0, 0):,}  ({target_counts.get(0, 0)/len(df)*100:.1f}%)")
print(f"   Delayed (1): {target_counts.get(1, 0):,}  ({target_counts.get(1, 0)/len(df)*100:.1f}%)")

# Missing values
n_missing = df.isnull().sum().sum()
print(f"\n2. Missing values: {n_missing}")

# Duplicates
n_dupes = df.duplicated().sum()
print(f"3. Duplicate rows: {n_dupes}")

# Feature breakdown
print(f"\n4. Feature breakdown:")
print(f"   Total columns:  {df.shape[1]}")
print(f"   Features:       {df.shape[1] - 1}")
print(f"   Target:         1 (delay)")

# Peek at first few rows
print(f"\n5. First 3 rows:")
df.head(3)

In [ ]:
# ================================================================
#  Identify column groups
# ================================================================
# Target
TARGET = 'delay'

# 6 categorical columns that need one-hot encoding
CATEGORICAL_COLS = [
    'route_description',       # e.g., "Pune-Mumbai", "Delhi-Jaipur"
    'origin_description',      # origin city name
    'destination_description', # destination city name
    'fuel_type',               # Diesel, CNG, etc.
    'gender',                  # driver gender
    'driving_style',           # Cautious, Normal, Aggressive
]

# 3 columns that are already binary or ordinal -- no encoding needed
BINARY_ORDINAL_COLS = [
    'accident',     # 0 or 1
    'ratings',      # ordinal 1-5
    'is_midnight',  # 0 or 1
]

# Everything else (excluding target and categoricals) is continuous
ALL_NON_FEATURE = [TARGET] + CATEGORICAL_COLS + BINARY_ORDINAL_COLS
CONTINUOUS_COLS = [c for c in df.columns if c not in ALL_NON_FEATURE]

print(f"Categorical (to one-hot encode): {len(CATEGORICAL_COLS)}")
for c in CATEGORICAL_COLS:
    n_unique = df[c].nunique()
    print(f"  - {c}: {n_unique} unique values")

print(f"\nBinary / Ordinal (keep as-is):   {len(BINARY_ORDINAL_COLS)}")
print(f"Continuous (to scale):           {len(CONTINUOUS_COLS)}")
print(f"Target:                          1 ({TARGET})")
print(f"\nTotal:  {len(CATEGORICAL_COLS)} + {len(BINARY_ORDINAL_COLS)} + {len(CONTINUOUS_COLS)} + 1 = {df.shape[1]}")

## 3. Train / Validation / Test Split

We use a **70 / 15 / 15** split with **stratification** on the `delay` target.

**Why stratified?** If 55% of trips are on-time and 45% are delayed, every split should preserve that ratio. Without stratification, one split might end up with 60% on-time by random chance, skewing the metrics.

**Why three splits, not two?**
- **Training set (70%)** — the model learns from this.
- **Validation set (15%)** — we tune and compare models on this. All three models are evaluated on the *same* validation set so comparisons are fair.
- **Test set (15%)** — touched only *once*, at the very end, to report the final unbiased performance of the winning model.

In [ ]:
# -- Separate features and target --
X = df.drop(columns=[TARGET])
y = df[TARGET]

# -- First split: 70% train, 30% temp --
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

# -- Second split: 50/50 of temp -> 15% validation + 15% test --
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print("=" * 55)
print("  SPLIT SUMMARY")
print("=" * 55)
for name, Xs, ys in [("Train", X_train, y_train),
                      ("Validation", X_val, y_val),
                      ("Test", X_test, y_test)]:
    pct_delayed = ys.mean() * 100
    print(f"  {name:12s}: {Xs.shape[0]:,} rows  |  Delayed: {pct_delayed:.1f}%")

print(f"\n  Total: {len(X_train) + len(X_val) + len(X_test):,} rows")

## 4. Preprocessing — One-Hot Encoding + Standard Scaling

Two critical rules to avoid **data leakage**:

1. **Fit only on training data.** The encoder and scaler learn their vocabularies / statistics from the training set alone.
2. **Transform all three splits** using the fitted encoder and scaler.

If you fit the scaler on the full dataset (including validation and test), the model indirectly "sees" information from those sets during training. This inflates your metrics and gives you a nasty surprise in production.

In [ ]:
# ================================================================
#  Step 4a: One-Hot Encode categorical columns
# ================================================================
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore', drop='first')

# Fit on training data only
encoder.fit(X_train[CATEGORICAL_COLS])

# Transform all splits
X_train_cat = pd.DataFrame(
    encoder.transform(X_train[CATEGORICAL_COLS]),
    columns=encoder.get_feature_names_out(CATEGORICAL_COLS),
    index=X_train.index
)
X_val_cat = pd.DataFrame(
    encoder.transform(X_val[CATEGORICAL_COLS]),
    columns=encoder.get_feature_names_out(CATEGORICAL_COLS),
    index=X_val.index
)
X_test_cat = pd.DataFrame(
    encoder.transform(X_test[CATEGORICAL_COLS]),
    columns=encoder.get_feature_names_out(CATEGORICAL_COLS),
    index=X_test.index
)

print(f"One-hot encoded columns: {len(CATEGORICAL_COLS)} original -> {X_train_cat.shape[1]} encoded")
print(f"\nEncoded feature names (first 15):")
for i, col in enumerate(encoder.get_feature_names_out(CATEGORICAL_COLS)[:15], 1):
    print(f"  {i:2d}. {col}")
if X_train_cat.shape[1] > 15:
    print(f"  ... and {X_train_cat.shape[1] - 15} more")

In [ ]:
# ================================================================
#  Step 4b: Standard-scale continuous columns
# ================================================================
scaler = StandardScaler()

# Fit on training data only
scaler.fit(X_train[CONTINUOUS_COLS])

# Transform all splits
X_train_cont = pd.DataFrame(
    scaler.transform(X_train[CONTINUOUS_COLS]),
    columns=CONTINUOUS_COLS,
    index=X_train.index
)
X_val_cont = pd.DataFrame(
    scaler.transform(X_val[CONTINUOUS_COLS]),
    columns=CONTINUOUS_COLS,
    index=X_val.index
)
X_test_cont = pd.DataFrame(
    scaler.transform(X_test[CONTINUOUS_COLS]),
    columns=CONTINUOUS_COLS,
    index=X_test.index
)

print(f"Scaled {len(CONTINUOUS_COLS)} continuous features")
print(f"\nTraining set stats after scaling (should be ~mean=0, std=1):")
print(X_train_cont.describe().loc[['mean', 'std']].round(3).to_string())

In [ ]:
# ================================================================
#  Step 4c: Assemble final feature matrices
# ================================================================
# Combine: scaled continuous + one-hot categorical + binary/ordinal (as-is)

def assemble_features(X_orig, X_cont, X_cat, binary_cols):
    """Combine continuous, categorical, and binary/ordinal features into one DataFrame."""
    X_bin = X_orig[binary_cols].reset_index(drop=True)
    X_cont_reset = X_cont.reset_index(drop=True)
    X_cat_reset = X_cat.reset_index(drop=True)
    return pd.concat([X_cont_reset, X_cat_reset, X_bin], axis=1)

X_train_final = assemble_features(X_train, X_train_cont, X_train_cat, BINARY_ORDINAL_COLS)
X_val_final   = assemble_features(X_val,   X_val_cont,   X_val_cat,   BINARY_ORDINAL_COLS)
X_test_final  = assemble_features(X_test,  X_test_cont,  X_test_cat,  BINARY_ORDINAL_COLS)

# Reset y indices to match
y_train_arr = y_train.reset_index(drop=True)
y_val_arr   = y_val.reset_index(drop=True)
y_test_arr  = y_test.reset_index(drop=True)

print("Final feature matrix shapes:")
print(f"  X_train: {X_train_final.shape}")
print(f"  X_val:   {X_val_final.shape}")
print(f"  X_test:  {X_test_final.shape}")
print(f"\nTotal features per sample: {X_train_final.shape[1]}")

FEATURE_NAMES = list(X_train_final.columns)
print(f"\nFeature list saved ({len(FEATURE_NAMES)} features)")

In [ ]:
# -- Save preprocessing artifacts for Lab E --
os.makedirs('artifacts', exist_ok=True)

encoder_path = 'artifacts/encoder.pkl'
scaler_path  = 'artifacts/scaler.pkl'

joblib.dump(encoder, encoder_path)
joblib.dump(scaler, scaler_path)

print(f"Saved: {encoder_path}  ({os.path.getsize(encoder_path)/1024:.1f} KB)")
print(f"Saved: {scaler_path}   ({os.path.getsize(scaler_path)/1024:.1f} KB)")

## 5. Helper Functions

We define two reusable functions:

1. **`log_classification_metrics()`** — calculates Accuracy, Precision, Recall, F1, ROC-AUC and logs them to MLflow.
2. **`plot_confusion_matrix()`** — creates a heatmap, saves it as a PNG, and logs the image as an MLflow artifact.

These functions are called identically for every model, ensuring consistent tracking.

In [ ]:
def log_classification_metrics(y_true, y_pred, y_prob, model_name):
    """Calculate and log all classification metrics to MLflow.

    Parameters
    ----------
    y_true : array-like - ground truth labels (0 or 1)
    y_pred : array-like - predicted labels (0 or 1)
    y_prob : array-like - predicted probability of the positive class (delayed)
    model_name : str - display name for printing

    Returns
    -------
    dict - metric name -> value
    """
    metrics = {
        'accuracy':  accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred),
        'recall':    recall_score(y_true, y_pred),
        'f1_score':  f1_score(y_true, y_pred),
        'roc_auc':   roc_auc_score(y_true, y_prob),
    }

    # Log every metric to MLflow
    for name, value in metrics.items():
        mlflow.log_metric(name, round(value, 4))

    # Pretty-print results
    print(f"\n{'='*55}")
    print(f"  {model_name} -- Validation Results")
    print(f"{'='*55}")
    for name, value in metrics.items():
        print(f"  {name:12s}: {value:.4f}")

    return metrics

In [ ]:
def plot_confusion_matrix(y_true, y_pred, model_name):
    """Create, display, save, and log a confusion matrix heatmap.

    Parameters
    ----------
    y_true : array-like - ground truth labels
    y_pred : array-like - predicted labels
    model_name : str - used in the title and filename

    Returns
    -------
    str - file path of the saved PNG
    """
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['On-Time', 'Delayed'],
                yticklabels=['On-Time', 'Delayed'])
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    ax.set_title(f'{model_name} -- Confusion Matrix')
    plt.tight_layout()

    filepath = f'artifacts/{model_name.lower().replace(" ", "_")}_confusion_matrix.png'
    fig.savefig(filepath, dpi=100)
    mlflow.log_artifact(filepath)
    plt.show()

    return filepath


def plot_feature_importance(importances, feature_names, model_name, top_n=15):
    """Bar chart of top feature importances; saved and logged to MLflow.

    Parameters
    ----------
    importances : array-like - feature importance values from the model
    feature_names : list - corresponding feature names
    model_name : str - used in the title and filename
    top_n : int - how many top features to display

    Returns
    -------
    str - file path of the saved PNG
    """
    imp_df = pd.DataFrame({
        'feature': feature_names,
        'importance': importances
    }).sort_values('importance', ascending=False).head(top_n)

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(imp_df['feature'][::-1], imp_df['importance'][::-1],
            color='#2563eb', edgecolor='white')
    ax.set_xlabel('Importance')
    ax.set_title(f'{model_name} -- Top {top_n} Feature Importances', fontweight='bold')
    plt.tight_layout()

    filepath = f'artifacts/{model_name.lower().replace(" ", "_")}_feature_importance.png'
    fig.savefig(filepath, dpi=100)
    mlflow.log_artifact(filepath)
    plt.show()

    return filepath

## 6. Understanding Classification Metrics

Before we start training, let us understand each metric in **plain English**. Imagine FreshBasket reviews 100 truck trips:

| Metric | What It Means | FreshBasket Analogy |
|--------|--------------|---------------------|
| **Accuracy** | % of all predictions that are correct | Out of 100 trips, how many did we label correctly (delayed or on-time)? |
| **Precision** | Of all trips we *predicted* as delayed, how many were *actually* delayed? | If we flag 20 trips as delayed, how many truly were? Low precision = too many false alarms (₹8,000 wasted per unnecessary reroute). |
| **Recall** | Of all trips that were *actually* delayed, how many did we *catch*? | If 40 trips were truly delayed, how many did our model flag? Low recall = missed delays (angry customers, spoiled produce). |
| **F1 Score** | Harmonic mean of Precision and Recall — balances both | A single number that penalizes extremes. If precision is 0.95 but recall is 0.10, F1 will be low. |
| **ROC-AUC** | How well the model separates the two classes across all probability thresholds | 0.5 = random guessing, 1.0 = perfect separation. Useful when you have not yet decided on a classification threshold. |

> **Priya's rule of thumb:** "For our logistics use case, *recall matters more than precision*. A missed delay costs us ₹25,000 in spoiled goods and customer churn. A false alarm costs ₹8,000 for an unnecessary backup truck. We can tolerate some false alarms, but we cannot afford to miss delays."

---

## 7. Model 1 — Logistic Regression (Baseline)

Logistic Regression is the simplest classification model. It fits a **linear decision boundary** in feature space and outputs a probability via the sigmoid function.

**Why start here?** It is fast, interpretable, and gives us a *floor* for comparison. If a complex model cannot beat Logistic Regression by a meaningful margin, the complexity is not worth it.

In [ ]:
# ================================================================
#  Model 1: Logistic Regression
# ================================================================
all_results = []  # collect metrics from all models

with mlflow.start_run(run_name="Logistic_Regression"):

    # -- Tags --
    mlflow.set_tag("project", "truck-delay-classification")
    mlflow.set_tag("module", "M3")
    mlflow.set_tag("model_type", "Logistic Regression")

    # -- Hyperparameters --
    lr_params = {
        'C': 1.0,
        'solver': 'lbfgs',
        'max_iter': 1000,
        'random_state': 42,
    }
    for k, v in lr_params.items():
        mlflow.log_param(k, v)
    mlflow.log_param("n_features", X_train_final.shape[1])

    # -- Train --
    lr_model = LogisticRegression(**lr_params)
    lr_model.fit(X_train_final, y_train_arr)

    # -- Predict on validation set --
    y_pred_lr  = lr_model.predict(X_val_final)
    y_prob_lr  = lr_model.predict_proba(X_val_final)[:, 1]

    # -- Log metrics + confusion matrix --
    lr_metrics = log_classification_metrics(y_val_arr, y_pred_lr, y_prob_lr,
                                            "Logistic Regression")
    plot_confusion_matrix(y_val_arr, y_pred_lr, "Logistic Regression")

    # -- Classification report --
    print("\nClassification Report:")
    print(classification_report(y_val_arr, y_pred_lr,
                                target_names=['On-Time', 'Delayed']))

    # -- Log model artifact --
    mlflow.sklearn.log_model(lr_model, "model")
    print("Model logged to MLflow")

    lr_metrics['model'] = 'Logistic Regression'
    all_results.append(lr_metrics)

In [ ]:
# -- ROC Curve for Logistic Regression --
fig, ax = plt.subplots(figsize=(7, 5))
RocCurveDisplay.from_predictions(y_val_arr, y_prob_lr, ax=ax, name="Logistic Regression")
ax.plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Random (AUC = 0.5)')
ax.set_title('Logistic Regression -- ROC Curve', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('artifacts/logistic_regression_roc.png', dpi=100)
plt.show()

## 8. Model 2 — Random Forest (Ensemble)

Random Forest builds many decision trees on random subsets of the data and features, then **votes** on the final prediction. It handles non-linear relationships, automatically captures feature interactions, and is resistant to overfitting (compared to a single deep tree).

**Key parameters:**
- `n_estimators` — number of trees in the forest (more = slower but more stable)
- `max_depth` — how deep each tree can grow (deeper = more complex, risk of overfitting)
- `min_samples_split` — minimum samples required to split an internal node

In [ ]:
# ================================================================
#  Model 2: Random Forest
# ================================================================
with mlflow.start_run(run_name="Random_Forest"):

    # -- Tags --
    mlflow.set_tag("project", "truck-delay-classification")
    mlflow.set_tag("module", "M3")
    mlflow.set_tag("model_type", "Random Forest")

    # -- Hyperparameters --
    rf_params = {
        'n_estimators': 200,
        'max_depth': 15,
        'min_samples_split': 10,
        'min_samples_leaf': 4,
        'random_state': 42,
        'n_jobs': -1,
    }
    for k, v in rf_params.items():
        mlflow.log_param(k, v)
    mlflow.log_param("n_features", X_train_final.shape[1])

    # -- Train --
    rf_model = RandomForestClassifier(**rf_params)
    rf_model.fit(X_train_final, y_train_arr)

    # -- Predict on validation set --
    y_pred_rf = rf_model.predict(X_val_final)
    y_prob_rf = rf_model.predict_proba(X_val_final)[:, 1]

    # -- Log metrics + confusion matrix --
    rf_metrics = log_classification_metrics(y_val_arr, y_pred_rf, y_prob_rf,
                                            "Random Forest")
    plot_confusion_matrix(y_val_arr, y_pred_rf, "Random Forest")

    # -- Classification report --
    print("\nClassification Report:")
    print(classification_report(y_val_arr, y_pred_rf,
                                target_names=['On-Time', 'Delayed']))

    # -- Feature importance --
    plot_feature_importance(rf_model.feature_importances_, FEATURE_NAMES,
                           "Random Forest", top_n=15)

    # -- Log model artifact --
    mlflow.sklearn.log_model(rf_model, "model")
    print("Model logged to MLflow")

    rf_metrics['model'] = 'Random Forest'
    all_results.append(rf_metrics)

## 9. Model 3 — XGBoost (Gradient Boosting)

XGBoost builds trees **sequentially** — each new tree tries to correct the mistakes of the previous one. This "boosting" approach often outperforms Random Forest, especially on structured/tabular data.

**Key parameters:**
- `learning_rate` — step size for each boosting round (lower = more rounds needed but better generalization)
- `n_estimators` — number of boosting rounds
- `max_depth` — depth of each tree (shallower = less overfitting)
- `subsample` — fraction of training rows used per tree (< 1.0 adds randomness, reduces overfitting)

> **Why XGBoost is expected to win:** It excels on medium-sized tabular datasets with mixed feature types — exactly what we have. The sequential error-correction mechanism gives it an edge over Random Forest's independent-tree voting.

In [ ]:
# ================================================================
#  Model 3: XGBoost
# ================================================================
with mlflow.start_run(run_name="XGBoost"):

    # -- Tags --
    mlflow.set_tag("project", "truck-delay-classification")
    mlflow.set_tag("module", "M3")
    mlflow.set_tag("model_type", "XGBoost")

    # -- Hyperparameters --
    xgb_params = {
        'learning_rate': 0.1,
        'n_estimators': 300,
        'max_depth': 6,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
        'reg_alpha': 0.1,          # L1 regularization
        'reg_lambda': 1.0,         # L2 regularization
        'random_state': 42,
        'eval_metric': 'logloss',
        'use_label_encoder': False,
    }
    for k, v in xgb_params.items():
        mlflow.log_param(k, v)
    mlflow.log_param("n_features", X_train_final.shape[1])

    # -- Train --
    xgb_model = xgb.XGBClassifier(**xgb_params)
    xgb_model.fit(
        X_train_final, y_train_arr,
        eval_set=[(X_val_final, y_val_arr)],
        verbose=False
    )

    # -- Predict on validation set --
    y_pred_xgb = xgb_model.predict(X_val_final)
    y_prob_xgb = xgb_model.predict_proba(X_val_final)[:, 1]

    # -- Log metrics + confusion matrix --
    xgb_metrics = log_classification_metrics(y_val_arr, y_pred_xgb, y_prob_xgb,
                                              "XGBoost")
    plot_confusion_matrix(y_val_arr, y_pred_xgb, "XGBoost")

    # -- Classification report --
    print("\nClassification Report:")
    print(classification_report(y_val_arr, y_pred_xgb,
                                target_names=['On-Time', 'Delayed']))

    # -- Feature importance --
    plot_feature_importance(xgb_model.feature_importances_, FEATURE_NAMES,
                           "XGBoost", top_n=15)

    # -- Log model artifact --
    mlflow.xgboost.log_model(xgb_model, "model")
    print("Model logged to MLflow")

    xgb_metrics['model'] = 'XGBoost'
    all_results.append(xgb_metrics)

In [ ]:
# -- XGBoost training history: plot validation loss over boosting rounds --
eval_results = xgb_model.evals_result()

if eval_results and 'validation_0' in eval_results:
    epochs = len(eval_results['validation_0']['logloss'])
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(range(epochs), eval_results['validation_0']['logloss'],
            color='#2563eb', linewidth=2)
    ax.set_xlabel('Boosting Round')
    ax.set_ylabel('Log Loss (Validation)')
    ax.set_title('XGBoost -- Validation Loss per Boosting Round', fontweight='bold')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('artifacts/xgboost_learning_curve.png', dpi=100)
    plt.show()
    print(f"Final validation log loss: {eval_results['validation_0']['logloss'][-1]:.4f}")
else:
    print("No eval results available -- learning curve skipped")

## 10. Model Comparison

Now we compare all three models **side-by-side** on the same validation set. The model with the highest **F1 Score** will be our winner, since F1 balances precision (avoiding false alarms) and recall (catching real delays).

> Priya: "Show Arjun the comparison table — he needs to justify the model choice to the logistics director at the next sprint review."

---

In [ ]:
# ================================================================
#  Model Comparison Table
# ================================================================
comparison_df = pd.DataFrame(all_results)
comparison_df = comparison_df[['model', 'accuracy', 'precision', 'recall', 'f1_score', 'roc_auc']]
comparison_df = comparison_df.sort_values('f1_score', ascending=False).reset_index(drop=True)

print("=" * 70)
print("  MODEL COMPARISON (Validation Set)")
print("=" * 70)
print(comparison_df.to_string(index=False, float_format='{:.4f}'.format))

# Highlight the winner
best_model_name = comparison_df.iloc[0]['model']
best_f1 = comparison_df.iloc[0]['f1_score']
print(f"\n  Winner: {best_model_name} (F1 = {best_f1:.4f})")

In [ ]:
# -- Side-by-side metric comparison chart --
metrics_to_plot = ['accuracy', 'precision', 'recall', 'f1_score', 'roc_auc']
x = np.arange(len(metrics_to_plot))
width = 0.25
colors = ['#64748b', '#2563eb', '#10b981']

fig, ax = plt.subplots(figsize=(12, 6))

for i, (_, row) in enumerate(comparison_df.iterrows()):
    values = [row[m] for m in metrics_to_plot]
    offset = (i - 1) * width
    ax.bar(x + offset, values, width, label=row['model'], color=colors[i],
           edgecolor='white', alpha=0.85)

ax.set_ylabel('Score')
ax.set_title('Model Comparison -- All Metrics (Validation Set)', fontweight='bold', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(['Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC-AUC'])
ax.legend()
ax.set_ylim(0, 1.05)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('artifacts/model_comparison.png', dpi=100)
plt.show()

In [ ]:
# -- Overlay ROC Curves for all 3 models --
fig, ax = plt.subplots(figsize=(8, 6))

RocCurveDisplay.from_predictions(y_val_arr, y_prob_lr,  ax=ax, name="Logistic Regression")
RocCurveDisplay.from_predictions(y_val_arr, y_prob_rf,  ax=ax, name="Random Forest")
RocCurveDisplay.from_predictions(y_val_arr, y_prob_xgb, ax=ax, name="XGBoost")

ax.plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Random (AUC = 0.5)')
ax.set_title('ROC Curves -- All Models', fontweight='bold', fontsize=14)
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig('artifacts/roc_curves_overlay.png', dpi=100)
plt.show()

## 11. Final Evaluation on Held-Out Test Set

The test set has been **completely untouched** until now. We evaluate the best model (XGBoost) on it to get an **unbiased estimate** of real-world performance.

> **Why not just use the validation set score?** Because we made model-selection decisions based on validation performance. That means the validation score is slightly optimistic. The test set gives us a number we can quote to stakeholders with confidence.

---

In [ ]:
# ================================================================
#  Final Evaluation: Best Model on Test Set
# ================================================================
# Select the best model (XGBoost based on validation F1)
best_model = xgb_model
BEST_MODEL_NAME = "XGBoost"

with mlflow.start_run(run_name="XGBoost_FINAL_TEST"):

    mlflow.set_tag("project", "truck-delay-classification")
    mlflow.set_tag("module", "M3")
    mlflow.set_tag("evaluation", "final_test_set")

    # -- Predict on TEST set --
    y_pred_test = best_model.predict(X_test_final)
    y_prob_test = best_model.predict_proba(X_test_final)[:, 1]

    # -- Log test metrics --
    test_metrics = {
        'test_accuracy':  accuracy_score(y_test_arr, y_pred_test),
        'test_precision': precision_score(y_test_arr, y_pred_test),
        'test_recall':    recall_score(y_test_arr, y_pred_test),
        'test_f1_score':  f1_score(y_test_arr, y_pred_test),
        'test_roc_auc':   roc_auc_score(y_test_arr, y_prob_test),
    }
    for k, v in test_metrics.items():
        mlflow.log_metric(k, round(v, 4))

    print("=" * 55)
    print(f"  {BEST_MODEL_NAME} -- FINAL TEST SET Results")
    print("=" * 55)
    for k, v in test_metrics.items():
        print(f"  {k:18s}: {v:.4f}")

    # -- Full classification report --
    print(f"\nClassification Report (Test Set):")
    print(classification_report(y_test_arr, y_pred_test,
                                target_names=['On-Time', 'Delayed']))

    # -- Confusion matrix --
    plot_confusion_matrix(y_test_arr, y_pred_test, f"{BEST_MODEL_NAME} (Test)")

In [ ]:
# -- Compare Validation vs Test performance --
print("=" * 60)
print(f"  {BEST_MODEL_NAME}: Validation vs Test")
print("=" * 60)
print(f"  {'Metric':18s} {'Validation':>12s} {'Test':>12s} {'Delta':>10s}")
print(f"  {'-'*54}")

val_metrics = xgb_metrics  # from the XGBoost training run
for metric in ['accuracy', 'precision', 'recall', 'f1_score', 'roc_auc']:
    v = val_metrics[metric]
    t = test_metrics[f'test_{metric}']
    delta = t - v
    sign = "+" if delta >= 0 else ""
    print(f"  {metric:18s} {v:12.4f} {t:12.4f} {sign}{delta:9.4f}")

print(f"\nSmall deltas (< 0.02) suggest the model generalizes well.")
print(f"Large drops would indicate overfitting to the validation set.")

## 12. Register Best Model in MLflow Model Registry

The **MLflow Model Registry** is a centralized store for managing model versions and their lifecycle stages:

| Stage | Meaning |
|-------|---------|
| **None** | Just logged, not reviewed |
| **Staging** | Under evaluation / QA testing |
| **Production** | Serving live traffic |
| **Archived** | Retired, kept for audit trail |

We will register our XGBoost model and move it to **Staging**. In Lab E, the Streamlit dashboard will load it from the registry.

---

In [ ]:
# ================================================================
#  Register the best model in MLflow Model Registry
# ================================================================
REGISTERED_MODEL_NAME = "truck-delay-classifier"

try:
    # Get the latest XGBoost run
    runs = mlflow.search_runs(
        filter_string="tags.model_type = 'XGBoost' AND tags.evaluation != 'final_test_set'",
        order_by=["metrics.f1_score DESC"],
        max_results=1
    )

    if len(runs) == 0:
        print("No XGBoost runs found. Skipping registration.")
    else:
        best_run_id = runs.iloc[0]['run_id']
        model_uri = f"runs:/{best_run_id}/model"

        print(f"Best XGBoost run ID: {best_run_id}")
        print(f"Model URI: {model_uri}")

        # Register the model
        result = mlflow.register_model(model_uri, REGISTERED_MODEL_NAME)
        print(f"\nRegistered: {REGISTERED_MODEL_NAME} v{result.version}")

        # Transition to Staging
        client = mlflow.tracking.MlflowClient()
        client.transition_model_version_stage(
            name=REGISTERED_MODEL_NAME,
            version=result.version,
            stage="Staging"
        )
        print(f"Transitioned to: Staging")
        print(f"\nModel Registry URI: {REGISTERED_MODEL_NAME} / v{result.version} / Staging")

except Exception as e:
    print(f"Model registration failed: {e}")
    print("This is OK if MLflow server is not running or Model Registry is not configured.")
    print("The model artifacts are still saved locally in the 'artifacts/' folder.")

In [ ]:
# -- Save the best model locally for Lab E --
model_path = 'artifacts/xgboost_model.pkl'
joblib.dump(best_model, model_path)

print(f"Saved: {model_path}  ({os.path.getsize(model_path)/1024:.1f} KB)")

# Save model metadata for downstream consumers
metadata = {
    'model_name': BEST_MODEL_NAME,
    'registered_name': REGISTERED_MODEL_NAME,
    'n_features': X_train_final.shape[1],
    'feature_names': FEATURE_NAMES,
    'categorical_cols': CATEGORICAL_COLS,
    'continuous_cols': CONTINUOUS_COLS,
    'binary_ordinal_cols': BINARY_ORDINAL_COLS,
    'test_metrics': {k: round(v, 4) for k, v in test_metrics.items()},
    'training_rows': len(X_train_final),
    'python_version': '3.12.10',
}

metadata_path = 'artifacts/model_metadata.json'
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"Saved: {metadata_path}")

# Summary of all artifacts
print(f"\nAll artifacts in artifacts/ directory:")
for fname in sorted(os.listdir('artifacts')):
    fpath = os.path.join('artifacts', fname)
    size_kb = os.path.getsize(fpath) / 1024
    print(f"  {fname:50s} ({size_kb:.1f} KB)")

## 13. Upload Artifacts to S3

Lab E (Streamlit Dashboard + Batch Scoring) will load the model, encoder, and scaler from S3. We upload the key artifacts now.

---

In [ ]:
# ================================================================
#  Upload artifacts to S3
# ================================================================
artifacts_to_upload = {
    'artifacts/xgboost_model.pkl':     f'{S3_MODEL_DIR}xgboost_model.pkl',
    'artifacts/encoder.pkl':           f'{S3_MODEL_DIR}encoder.pkl',
    'artifacts/scaler.pkl':            f'{S3_MODEL_DIR}scaler.pkl',
    'artifacts/model_metadata.json':   f'{S3_MODEL_DIR}model_metadata.json',
}

try:
    s3 = boto3.client('s3')
    print("Uploading artifacts to S3...\n")

    for local_path, s3_key in artifacts_to_upload.items():
        s3.upload_file(local_path, S3_BUCKET, s3_key)
        size_kb = os.path.getsize(local_path) / 1024
        print(f"  Uploaded: s3://{S3_BUCKET}/{s3_key}  ({size_kb:.1f} KB)")

    print(f"\nAll {len(artifacts_to_upload)} artifacts uploaded to S3")

except Exception as e:
    print(f"S3 upload failed: {e}")
    print("Artifacts are still available locally in the artifacts/ folder.")
    print("You can upload them manually later or configure AWS credentials.")

## 14. Summary & Artifacts Produced

### What We Covered

| Task | Key Takeaway |
|------|-------------|
| **Data split** | 70/15/15 stratified split avoids data leakage and gives an unbiased test evaluation |
| **Preprocessing** | OneHotEncoder + StandardScaler fitted on training data only |
| **Logistic Regression** | Simple baseline — fast and interpretable, but limited by linear decision boundary |
| **Random Forest** | Ensemble of independent trees — handles non-linearity, good out-of-the-box |
| **XGBoost** | Sequential boosting — best F1 and ROC-AUC; expected winner for tabular data |
| **MLflow tracking** | Every run logged with params, metrics, artifacts, and model objects |
| **Model Registry** | Best model registered and moved to Staging |
| **S3 upload** | Model + encoder + scaler + metadata ready for Lab E |

### Artifacts Produced

| File | Contents | Used By |
|------|----------|---------|
| `artifacts/xgboost_model.pkl` | Trained XGBoost classifier | Lab E (Streamlit + batch scoring) |
| `artifacts/encoder.pkl` | Fitted OneHotEncoder | Lab E (preprocessing new data) |
| `artifacts/scaler.pkl` | Fitted StandardScaler | Lab E (preprocessing new data) |
| `artifacts/model_metadata.json` | Feature names, metrics, config | Lab E (metadata display) |
| `artifacts/*_confusion_matrix.png` | Confusion matrix plots | MLflow artifacts |
| `artifacts/*_feature_importance.png` | Feature importance charts | MLflow artifacts |

### MLflow Experiment Runs

Open your MLflow UI at `http://<EC2_PUBLIC_IP>:5000` and navigate to the **truck-delay-classification** experiment. You should see **4 runs**:

1. `Logistic_Regression` — baseline
2. `Random_Forest` — ensemble
3. `XGBoost` — best model
4. `XGBoost_FINAL_TEST` — final test evaluation

> **[SCREENSHOT PLACEHOLDER]:** Take a screenshot of your MLflow UI showing all 4 runs with their metrics. This demonstrates that experiment tracking is working end-to-end.

---

### Try on Your Own

1. **Tune XGBoost hyperparameters** — try `learning_rate=0.05`, `n_estimators=500`, `max_depth=8`. Log the run to MLflow and compare with the original. Did the F1 score improve?
2. **Add a Gradient Boosting Classifier** from scikit-learn (`sklearn.ensemble.GradientBoostingClassifier`). How does it compare to XGBoost?
3. **Adjust the classification threshold** — instead of the default 0.5, try 0.4 (lower threshold = higher recall). Recompute precision and recall. At what threshold does recall hit 0.90?
4. **Try class weighting** — set `scale_pos_weight` in XGBoost to the ratio of on-time to delayed samples. Does this improve recall without destroying precision?
5. **Feature selection** — remove the bottom 10 features by importance and retrain. Does the model maintain its performance with fewer features?

---

**Next Lab:** Module 3, Lab E — Streamlit Dashboard & Batch Scoring Pipeline

In [ ]:
# -- Lab D Complete --
print("=" * 55)
print("  Lab D: Model Training with MLflow -- COMPLETE")
print("=" * 55)
print(f"\n  Models trained:    3 (LR, RF, XGBoost)")
print(f"  MLflow runs:       4 (3 training + 1 final test)")
print(f"  Best model:        XGBoost")
print(f"  Artifacts saved:   artifacts/ directory")
print(f"  Next step:         Lab E -- Streamlit + Batch Scoring")